# Projet IA – Suivi des expériences avec MLflow (DagsHub)

**Groupe 9 – Salome Bolivar** · Système de recommandation musicale, Jalon 2 v2

Ce notebook reprend `jalon2_v2_recommandation.ipynb` (dépôt projet-musical-IA) et enregistre chaque essai dans MLflow :
- données : écoutes réelles du *Taste Profile* du Million Song Dataset reliées à `spotify.csv` (20 000 utilisateurs) ;
- modèles : popularité, cosinus audio (système actuel), KNN item-item, SVD et EASE ;
- métriques du contrat d’optimisation : précision@5, rappel@5, hit rate@5 et temps d’inférence.

Chaque essai de validation (choix de k pour SVD et de λ pour EASE) et chaque résultat final sur le jeu de test devient un *run* MLflow.

**Pour l’exécuter (Google Colab ou Jupyter) :**
1. Copier son jeton d’accès DagsHub : https://dagshub.com/user/settings/tokens
2. *Exécution › Tout exécuter*, puis coller le jeton dans le champ affiché par la cellule « Connexion à DagsHub » et appuyer sur Entrée.
3. Ouvrir l’interface MLflow (lien affiché à la fin) : c’est le lien à remettre.

In [ ]:
%pip install -q "mlflow<3" scipy scikit-learn pandas

In [ ]:
import os, time
import numpy as np
import pandas as pd
import mlflow
from scipy import sparse
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import TruncatedSVD

GRAINE = 42
GITHUB_USER = "Salome2601"
DEPOT = "ProjetIA_mlflow_dagshub_2026"          # même nom sur GitHub et sur DagsHub
DAGSHUB_USER = "Salome2601"                      # nom d’utilisateur DagsHub
UTILISER_DAGSHUB = True                          # False : MLflow en local (dossier mlruns)
URI_MLFLOW = f"https://dagshub.com/{DAGSHUB_USER}/{DEPOT}.mlflow"   # serveur MLflow hébergé par DagsHub

## Connexion à DagsHub

MLflow envoie chaque essai au serveur MLflow hébergé par DagsHub, avec le nom d’utilisateur DagsHub et un **jeton d’accès** (configuration MLflow de DagsHub : `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`). Le jeton n’est jamais écrit dans le notebook :
- la cellule affiche un champ masqué : coller le jeton (https://dagshub.com/user/settings/tokens) puis appuyer sur Entrée ;
- ou bien l’enregistrer une fois pour toutes dans les **Secrets** de Colab (icône de clé), sous le nom `DAGSHUB_TOKEN` : la cellule le lit alors directement.

In [ ]:
if UTILISER_DAGSHUB:
    from getpass import getpass
    try:
        from google.colab import userdata          # Secrets de Colab (icône de clé)
        jeton = userdata.get("DAGSHUB_TOKEN")
    except Exception:
        jeton = os.environ.get("DAGSHUB_TOKEN") or getpass("Colle ton jeton DagsHub puis appuie sur Entrée : ")
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
    os.environ["MLFLOW_TRACKING_PASSWORD"] = jeton.strip()
    mlflow.set_tracking_uri(URI_MLFLOW)

try:
    mlflow.set_experiment("recommandation-jalon2-v2")
except Exception as erreur:
    raise RuntimeError("MLflow refuse la connexion : vérifie le jeton DagsHub (copié en entier) "
                       "et DAGSHUB_USER, puis relance cette cellule.") from erreur
print("Suivi MLflow :", mlflow.get_tracking_uri())

## 1. Données

`data/msd/` vient du script `scripts/preparer_donnees_msd.py` du projet (Taste Profile du Million Song Dataset, Bertin-Mahieux et al., 2011, relié à `spotify.csv`).

In [ ]:
URL_DONNEES = f"https://raw.githubusercontent.com/{GITHUB_USER}/{DEPOT}/main/data/msd/"
dossier = "data/msd/" if os.path.exists("data/msd/ecoutes.csv.gz") else URL_DONNEES
ecoutes = pd.read_csv(dossier + "ecoutes.csv.gz")
chansons = pd.read_csv(dossier + "chansons.csv")
print("Écoutes :", ecoutes.shape, "| chansons :", chansons.shape)

## 2. Nettoyage : plafond des écoutes et filtres (au moins 10 utilisateurs par chanson, 10 chansons par utilisateur)

In [ ]:
colonnes_audio = ["danceability", "energy", "loudness", "speechiness", "acousticness",
                  "instrumentalness", "liveness", "valence", "tempo"]

plafond = ecoutes["plays"].quantile(0.999)
ecoutes["plays"] = ecoutes["plays"].clip(upper=plafond)

MIN_USERS_PAR_CHANSON = 10
MIN_CHANSONS_PAR_USER = 10
while True:
    avant = len(ecoutes)
    nb_users_chanson = ecoutes.groupby("track_id")["user_id"].transform("size")
    ecoutes = ecoutes[nb_users_chanson >= MIN_USERS_PAR_CHANSON]
    nb_chansons_user = ecoutes.groupby("user_id")["track_id"].transform("size")
    ecoutes = ecoutes[nb_chansons_user >= MIN_CHANSONS_PAR_USER]
    if len(ecoutes) == avant:
        break

chansons = chansons[chansons["track_id"].isin(ecoutes["track_id"])].reset_index(drop=True)
print("Écoutes :", len(ecoutes), "| utilisateurs :", ecoutes["user_id"].nunique(), "| chansons :", len(chansons))

## 3. Séparation par utilisateur (80 % entraînement, 10 % validation, 10 % test) et chansons cachées (20 % par utilisateur)

In [ ]:
index_chanson = pd.Series(np.arange(len(chansons)), index=chansons["track_id"])
ecoutes["col"] = ecoutes["track_id"].map(index_chanson)
NB_CHANSONS = len(chansons)

rng = np.random.default_rng(GRAINE)
users = rng.permutation(ecoutes["user_id"].unique())
n = len(users)
users_train = users[: int(0.8 * n)]
users_val = users[int(0.8 * n): int(0.9 * n)]
users_test = users[int(0.9 * n):]


def matrice(df, users):
    ligne = pd.Series(np.arange(len(users)), index=users)
    lignes = df["user_id"].map(ligne).to_numpy()
    return sparse.csr_matrix((np.ones(len(df)), (lignes, df["col"].to_numpy())),
                             shape=(len(users), NB_CHANSONS))


def cacher_chansons(users, graine, part=0.2):
    df = ecoutes[ecoutes["user_id"].isin(users)].copy()
    df["hasard"] = np.random.default_rng(graine).random(len(df))
    rang = df.groupby("user_id")["hasard"].rank(method="first")
    nb = df.groupby("user_id")["hasard"].transform("size")
    df["cachee"] = rang <= np.maximum(1, np.round(part * nb))
    return matrice(df[~df["cachee"]], users), matrice(df[df["cachee"]], users)


X_train = matrice(ecoutes[ecoutes["user_id"].isin(users_train)], users_train)
profil_val, cachees_val = cacher_chansons(users_val, graine=1)
profil_test, cachees_test = cacher_chansons(users_test, graine=2)
print("Train :", len(users_train), "| validation :", len(users_val), "| test :", len(users_test))

## 4. Métriques du contrat et enregistrement dans MLflow

In [ ]:
def evaluer(recommander, profil, cachees, k=5):
    debut = time.perf_counter()
    scores = np.asarray(recommander(profil), dtype=float)
    scores[profil.nonzero()] = -np.inf
    top = np.argpartition(-scores, k, axis=1)[:, :k]
    duree = time.perf_counter() - debut
    bonnes = np.asarray(cachees[np.arange(top.shape[0])[:, None], top].sum(axis=1)).ravel()
    nb_cachees = np.asarray(cachees.sum(axis=1)).ravel()
    return {
        "precision_at_5": bonnes.mean() / k,
        "rappel_at_5": (bonnes / nb_cachees).mean(),
        "hit_rate_at_5": (bonnes > 0).mean(),
        "temps_ms_par_utilisateur": 1000 * duree / top.shape[0],
    }


resultats = []

def essai(nom, recommander, params, jeu="validation"):
    profil, cachees = (profil_val, cachees_val) if jeu == "validation" else (profil_test, cachees_test)
    metriques = evaluer(recommander, profil, cachees)
    with mlflow.start_run(run_name=f"{jeu} - {nom}"):
        mlflow.set_tag("jeu", jeu)
        mlflow.log_params({**params, "jeu": jeu, "graine": GRAINE, "nb_chansons": NB_CHANSONS,
                           "utilisateurs_train": X_train.shape[0]})
        mlflow.log_metrics(metriques)
    resultats.append({"run": f"{jeu} - {nom}", **metriques})
    return metriques

## 5. Modèles de référence et filtrage collaboratif (validation)

In [ ]:
# Popularité
popularite = np.asarray(X_train.sum(axis=0)).ravel()
def recommander_populaires(profil):
    return np.tile(popularite, (profil.shape[0], 1))
essai("Popularité", recommander_populaires, {"modele": "popularite"})

# Cosinus audio + popularité Spotify (système actuel, référence)
audio = normalize(StandardScaler().fit_transform(chansons[colonnes_audio]))
popularite_spotify = chansons["popularity"].to_numpy() / 100
def recommander_cosinus(profil):
    nb = np.asarray(profil.sum(axis=1))
    return 0.8 * ((profil @ audio) @ audio.T / nb) + 0.2 * popularite_spotify
essai("Cosinus audio", recommander_cosinus, {"modele": "cosinus_audio", "poids_popularite": 0.2})

# KNN item-item
vecteurs_chansons = normalize(X_train.T.tocsr())
sim_chansons = (vecteurs_chansons @ vecteurs_chansons.T).toarray()
np.fill_diagonal(sim_chansons, 0)
def recommander_knn(profil):
    return profil @ sim_chansons
essai("KNN item-item", recommander_knn, {"modele": "knn_item_item", "similarite": "cosinus"})

# SVD : choix de k
essais_svd = {}
for k in [16, 32, 64, 128]:
    V = TruncatedSVD(n_components=k, random_state=GRAINE).fit(X_train).components_
    essais_svd[k] = essai(f"SVD (k={k})", lambda p: (p @ V.T) @ V, {"modele": "svd", "k": k})
meilleur_k = max(essais_svd, key=lambda k: essais_svd[k]["precision_at_5"])
V_svd = TruncatedSVD(n_components=meilleur_k, random_state=GRAINE).fit(X_train).components_
def recommander_svd(profil):
    return (profil @ V_svd.T) @ V_svd

# EASE : choix de λ
def entrainer_ease(X, lam):
    G = (X.T @ X).toarray()
    G[np.diag_indices_from(G)] += lam
    P = np.linalg.inv(G)
    B = -P / np.diag(P)
    np.fill_diagonal(B, 0)
    return B

essais_ease = {}
for lam in [10, 50, 200, 1000]:
    B = entrainer_ease(X_train, lam)
    essais_ease[lam] = essai(f"EASE (λ={lam})", lambda p: p @ B, {"modele": "ease", "lambda": lam})
meilleur_lam = max(essais_ease, key=lambda lam: essais_ease[lam]["precision_at_5"])
B_ease = entrainer_ease(X_train, meilleur_lam)
def recommander_ease(profil):
    return profil @ B_ease

print("Meilleur k (SVD) :", meilleur_k, "| meilleur λ (EASE) :", meilleur_lam)

## 6. Résultats finaux sur le jeu de test

In [ ]:
modeles = {
    "Popularité": (recommander_populaires, {"modele": "popularite"}),
    "Cosinus audio": (recommander_cosinus, {"modele": "cosinus_audio", "poids_popularite": 0.2}),
    "KNN item-item": (recommander_knn, {"modele": "knn_item_item", "similarite": "cosinus"}),
    f"SVD (k={meilleur_k})": (recommander_svd, {"modele": "svd", "k": meilleur_k}),
    f"EASE (λ={meilleur_lam})": (recommander_ease, {"modele": "ease", "lambda": meilleur_lam}),
}
for nom, (fonction, params) in modeles.items():
    essai(nom, fonction, params, jeu="test")

tableau = pd.DataFrame(resultats).set_index("run")
tableau[tableau.index.str.startswith("test")].round(4)

In [ ]:
if UTILISER_DAGSHUB:
    print("Interface MLflow à remettre :", URI_MLFLOW)
else:
    print("Runs enregistrés en local :", mlflow.get_tracking_uri())